In [6]:
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np

ZIP_DIR = Path(r"C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\raw_zips")

THEFT_CODES = {"23A","23B","23C","23D","23E","23F","23G","23H"}

def read_csv_from_zip(zip_path: Path, filename: str, usecols=None):
    with zipfile.ZipFile(zip_path, "r") as zf:
        
        members = [m for m in zf.namelist() if m.lower().endswith(filename.lower())]
        if not members:
            raise FileNotFoundError(f"{filename} not found in {zip_path.name}")
        member = members[0]
        with zf.open(member) as f:
            return pd.read_csv(f, usecols=usecols)

def find_col(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"Cannot find column among {candidates}. Available: {list(cols)[:30]}")

def summarize_one_zip(zip_path: Path):
    # 1) OFFENSE_TYPE
    offtype = read_csv_from_zip(zip_path, "NIBRS_OFFENSE_TYPE.csv")


    col_offtype_code = find_col(offtype.columns, ["offense_code"])
    col_offtype_name = find_col(offtype.columns, ["offense_name", "offense_type_name", "offense", "name"])


    col_offtype_id = None
    try:
        col_offtype_id = find_col(offtype.columns, ["offense_type_id"])
    except:
        pass

    # 2) OFFENSE
    offense = read_csv_from_zip(zip_path, "NIBRS_OFFENSE.csv")
    col_off_incident = find_col(offense.columns, ["incident_id"])

    
    if any(c.lower() == "offense_code" for c in offense.columns):
        col_off_code = find_col(offense.columns, ["offense_code"])
        offense[col_off_code] = offense[col_off_code].astype(str)
        theft_off = offense[offense[col_off_code].isin(THEFT_CODES)].copy()

    
    else:
        col_off_typeid = find_col(offense.columns, ["offense_type_id"])
        if col_offtype_id is None:
            raise KeyError(
                f"{zip_path.name}: OFFENSE has offense_type_id but OFFENSE_TYPE has no offense_type_id column"
            )

        offense[col_off_typeid] = pd.to_numeric(offense[col_off_typeid], errors="coerce")
        offtype[col_offtype_id] = pd.to_numeric(offtype[col_offtype_id], errors="coerce")

        offense = offense.dropna(subset=[col_off_typeid]).copy()
        offtype = offtype.dropna(subset=[col_offtype_id]).copy()

        offense[col_off_typeid] = offense[col_off_typeid].astype(int)
        offtype[col_offtype_id] = offtype[col_offtype_id].astype(int)

        offense2 = offense.merge(
            offtype[[col_offtype_id, col_offtype_code, col_offtype_name]],
            left_on=col_off_typeid,
            right_on=col_offtype_id,
            how="left"
        )

        offense2[col_offtype_code] = offense2[col_offtype_code].astype(str)
        theft_off = offense2[offense2[col_offtype_code].isin(THEFT_CODES)].copy()

    theft_incidents = theft_off[[col_off_incident]].drop_duplicates()

    # 3) INCIDENT
    incident = read_csv_from_zip(zip_path, "NIBRS_incident.csv")
    col_inc_incident = find_col(incident.columns, ["incident_id"])
    col_inc_agency   = find_col(incident.columns, ["agency_id"])
    col_inc_date     = find_col(incident.columns, ["incident_date", "date"])

    # 4) merge -> events
    events = theft_incidents.merge(
        incident[[col_inc_incident, col_inc_agency, col_inc_date]],
        left_on=col_off_incident, right_on=col_inc_incident, how="inner"
    ).rename(columns={
        col_inc_incident: "incident_id",
        col_inc_agency: "agency_id",
        col_inc_date: "date"
    })

    events["date"] = pd.to_datetime(events["date"], errors="coerce")
    events = events.dropna(subset=["date", "agency_id", "incident_id"]).drop_duplicates(subset=["incident_id"])

    
    n_theft = len(events)

  
    n_agencies = events["agency_id"].nunique()

    
    events["week"] = events["date"].dt.to_period("W").dt.start_time
    weekly = events.groupby(["agency_id", "week"]).size().reset_index(name="crime_count")

    if len(weekly) == 0:
        n_panel = 0
        date_min, date_max = None, None
    else:
        date_min = events["date"].min().date()
        date_max = events["date"].max().date()
        weeks = pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="7D")
        n_panel = n_agencies * len(weeks)

    return {
        "zip": zip_path.name,
        "theft_incidents": n_theft,
        "agencies": n_agencies,
        "agency_week_panel_rows": int(n_panel),
        "date_min": str(date_min) if date_min else None,
        "date_max": str(date_max) if date_max else None,
    }

rows = []
fails = []

for zp in sorted(ZIP_DIR.glob("nibrs_incidents_*.zip")):
    try:
        result = summarize_one_zip(zp)
        rows.append(result)
        print("[OK]  ", zp.name)
    except Exception as e:
        fails.append((zp.name, repr(e)))
        print("[FAIL]", zp.name, "->", repr(e))

summary = pd.DataFrame(rows).sort_values(["zip"])
display(summary)

[OK]   nibrs_incidents_CA_2024.zip
[OK]   nibrs_incidents_IL_2019.zip
[OK]   nibrs_incidents_IL_2020.zip


C:\Users\23363\AppData\Local\Temp\ipykernel_135940\1143339044.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  events["date"] = pd.to_datetime(events["date"], errors="coerce")
C:\Users\23363\AppData\Local\Temp\ipykernel_135940\1143339044.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  events["date"] = pd.to_datetime(events["date"], errors="coerce")


[OK]   nibrs_incidents_IL_2021.zip
[OK]   nibrs_incidents_IL_2022.zip


C:\Users\23363\AppData\Local\Temp\ipykernel_135940\1143339044.py:18: DtypeWarning: Columns (0: cleared_except_date) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(f, usecols=usecols)


[OK]   nibrs_incidents_IL_2023.zip


C:\Users\23363\AppData\Local\Temp\ipykernel_135940\1143339044.py:18: DtypeWarning: Columns (0: cleared_except_date) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(f, usecols=usecols)


[OK]   nibrs_incidents_IL_2024.zip
[OK]   nibrs_incidents_IN_2024.zip
[OK]   nibrs_incidents_MI_2024.zip


C:\Users\23363\AppData\Local\Temp\ipykernel_135940\1143339044.py:18: DtypeWarning: Columns (0: cleared_except_date) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(f, usecols=usecols)


[OK]   nibrs_incidents_NY_2024.zip


,zip,theft_incidents,agencies,agency_week_panel_rows,date_min,date_max
0,nibrs_incidents_CA_2024.zip,352478,564,29892,2024-01-01,2024-12-31
1,nibrs_incidents_IL_2019.zip,3578,1,53,2019-01-01,2019-12-31
2,nibrs_incidents_IL_2020.zip,3266,7,371,2020-01-01,2020-12-31
3,nibrs_incidents_IL_2021.zip,59994,299,15847,2021-01-01,2021-12-31
4,nibrs_incidents_IL_2022.zip,122476,467,24751,2022-01-01,2022-12-31
5,nibrs_incidents_IL_2023.zip,134812,556,29468,2023-01-01,2023-12-31
6,nibrs_incidents_IL_2024.zip,139517,598,31694,2024-01-01,2024-12-31
7,nibrs_incidents_IN_2024.zip,64408,219,11607,2024-01-01,2024-12-31
8,nibrs_incidents_MI_2024.zip,96875,583,30899,2024-01-01,2024-12-31
9,nibrs_incidents_NY_2024.zip,239257,191,10123,2024-01-01,2024-12-31


In [8]:
import json
import shutil
import zipfile
from pathlib import Path

BASE_DIR = Path(r"C:\Users\23363\Desktop\IT5006\Project\NIBRS\data")
RAW_ZIPS_DIR = BASE_DIR / "raw_zips"
DATASETS_DIR = BASE_DIR / "datasets"

# ===== 直接按文件名白名单保留 =====
TEMPORAL_ZIPS = {
    "nibrs_incidents_IL_2021.zip",
    "nibrs_incidents_IL_2022.zip",
    "nibrs_incidents_IL_2023.zip",
    "nibrs_incidents_IL_2024.zip",
}

GEOGRAPHIC_ZIPS = {
    "nibrs_incidents_IL_2024.zip",
    "nibrs_incidents_IN_2024.zip",
    "nibrs_incidents_MI_2024.zip",
    "nibrs_incidents_CA_2024.zip",
    "nibrs_incidents_NY_2024.zip",
}

ALL_KEEP = TEMPORAL_ZIPS | GEOGRAPHIC_ZIPS

NEEDED = {
    "NIBRS_incident.csv": "nibrs_incident.csv",
    "NIBRS_OFFENSE.csv": "nibrs_offense.csv",
    "NIBRS_OFFENSE_TYPE.csv": "nibrs_offense_type.csv",
    "agencies.csv": "agencies.csv",   # 可选
}

def find_file_case_insensitive(root: Path, target_name: str):
    for p in root.rglob("*"):
        if p.is_file() and p.name.lower() == target_name.lower():
            return p
    return None

def extract_one_zip(zip_path: Path, dest_root: Path, tag: str):
    # 例：nibrs_incidents_IL_2024.zip -> IL_2024
    stem = zip_path.stem  # nibrs_incidents_IL_2024
    parts = stem.split("_")
    state = parts[-2]
    year = parts[-1]
    dataset_name = f"{state}_{year}"

    dataset_dir = dest_root / tag / dataset_name
    raw_dir = dataset_dir / "raw"
    tables_dir = dataset_dir / "tables"
    tmp_dir = dataset_dir / "_tmp"

    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    tmp_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp_dir)

    shutil.copytree(tmp_dir, raw_dir)

    tables_dir.mkdir(parents=True, exist_ok=True)
    copied = {}

    for canonical_name, alias in NEEDED.items():
        src = find_file_case_insensitive(tmp_dir, alias)
        if src is None:
            if canonical_name in ["NIBRS_incident.csv", "NIBRS_OFFENSE.csv", "NIBRS_OFFENSE_TYPE.csv"]:
                raise FileNotFoundError(f"{zip_path.name} 缺少必须文件: {alias}")
            else:
                continue
        dst = tables_dir / canonical_name
        shutil.copy2(src, dst)
        copied[canonical_name] = str(dst)

    manifest = {
        "tag": tag,
        "source_zip": zip_path.name,
        "dataset_name": dataset_name,
        "tables": copied,
    }
    with open(dataset_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"[OK] {tag:10s} -> {dataset_dir}")

# ===== main =====
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

zips = sorted(RAW_ZIPS_DIR.glob("*.zip"))
print(f"Found zip files: {len(zips)}\n")

print("=== Will process ===")
for zp in zips:
    if zp.name in ALL_KEEP:
        print(" ", zp.name)

print("\n=== Skipped ===")
for zp in zips:
    if zp.name not in ALL_KEEP:
        print(" ", zp.name, "-> 不在保留计划中")

for zp in zips:
    if zp.name in TEMPORAL_ZIPS:
        extract_one_zip(zp, DATASETS_DIR, "temporal")
    if zp.name in GEOGRAPHIC_ZIPS:
        extract_one_zip(zp, DATASETS_DIR, "geographic")

print("\nDone.")

Found zip files: 10

=== Will process ===
  nibrs_incidents_CA_2024.zip
  nibrs_incidents_IL_2021.zip
  nibrs_incidents_IL_2022.zip
  nibrs_incidents_IL_2023.zip
  nibrs_incidents_IL_2024.zip
  nibrs_incidents_IN_2024.zip
  nibrs_incidents_MI_2024.zip
  nibrs_incidents_NY_2024.zip

=== Skipped ===
  nibrs_incidents_IL_2019.zip -> 不在保留计划中
  nibrs_incidents_IL_2020.zip -> 不在保留计划中
[OK] geographic -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\geographic\CA_2024
[OK] temporal   -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\temporal\IL_2021
[OK] temporal   -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\temporal\IL_2022
[OK] temporal   -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\temporal\IL_2023
[OK] temporal   -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\temporal\IL_2024
[OK] geographic -> C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets\geographic\IL_2024
[OK] geographic -> C:\Users\23363\Desktop\IT5006\Pr

In [9]:
from pathlib import Path

DATASETS_DIR = Path(r"C:\Users\23363\Desktop\IT5006\Project\NIBRS\data\datasets")

print("=== temporal ===")
for p in sorted((DATASETS_DIR / "temporal").glob("*")):
    print(p.name)

print("\n=== geographic ===")
for p in sorted((DATASETS_DIR / "geographic").glob("*")):
    print(p.name)

=== temporal ===
IL_2021
IL_2022
IL_2023
IL_2024

=== geographic ===
CA_2024
IL_2024
IN_2024
MI_2024
NY_2024
